In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
df = pd.read_csv("../data/ricewheatprice.csv")

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [3]:
df.head()

,Year,Month,Price_wheat_ton,Price_rice_ton,Price_corn_ton,Inflation_rate,Price_wheat_ton_infl,Price_rice_ton_infl,Price_corn_ton_infl
0,1992,Feb,170.12,278.25,113.62,89.59,322.53,527.53,215.41
1,1992,Mar,161.44,277.20,117.00,89.59,306.07,525.54,221.82
2,1992,Apr,153.07,278.00,108.52,89.59,290.21,527.06,205.74
3,1992,May,139.72,274.00,109.64,89.59,264.90,519.48,207.87
4,1992,Jun,140.36,268.80,110.90,89.59,266.11,509.62,210.26


In [4]:
print("Shape")
print(df.shape)

print("\nColumns")
print(df.columns.tolist())

print("\nInformation")
df.info()

print("\nMissing Values")
print(df.isnull().sum())

print("\nStatistics")
display(df.describe())

Shape
(360, 9)

Columns
['Year', 'Month', 'Price_wheat_ton', 'Price_rice_ton', 'Price_corn_ton', 'Inflation_rate', 'Price_wheat_ton_infl', 'Price_rice_ton_infl', 'Price_corn_ton_infl']

Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 360 entries, 0 to 359
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Year                  360 non-null    int64  
 1   Month                 360 non-null    object 
 2   Price_wheat_ton       359 non-null    float64
 3   Price_rice_ton        360 non-null    float64
 4   Price_corn_ton        360 non-null    float64
 5   Inflation_rate        359 non-null    float64
 6   Price_wheat_ton_infl  359 non-null    float64
 7   Price_rice_ton_infl   359 non-null    float64
 8   Price_corn_ton_infl   359 non-null    float64
dtypes: float64(7), int64(1), object(1)
memory usage: 25.4+ KB

Missing Values
Year                    0
Month                   0
Price_

,Year,Price_wheat_ton,Price_rice_ton,Price_corn_ton,Inflation_rate,Price_wheat_ton_infl,Price_rice_ton_infl,Price_corn_ton_infl
count,360.000000,359.000000,360.000000,360.000000,359.000000,359.000000,359.000000,359.000000
mean,2006.583333,185.302869,364.105611,155.502750,36.316685,241.726769,474.038384,201.211086
std,8.671906,64.985279,131.367591,62.611439,26.605378,65.234300,133.776144,61.727752
min,1992.000000,85.300000,163.750000,75.270000,-1.290000,136.220000,246.020000,116.280000
25%,1999.000000,137.315000,261.625000,104.192500,12.370000,193.010000,397.500000,159.015000
50%,2007.000000,175.270000,344.675000,149.745000,28.300000,228.310000,455.540000,186.020000
75%,2014.000000,220.260000,444.250000,176.652500,59.700000,275.835000,540.180000,220.940000
max,2022.000000,419.610000,907.000000,333.050000,89.590000,518.470000,1120.690000,385.910000


In [5]:
# Remove duplicate rows
df = df.drop_duplicates()

# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
)

# Convert numeric columns
numeric_columns = [
    "Price_wheat_ton",
    "Price_rice_ton",
    "Price_corn_ton",
    "Inflation_rate",
    "Price_wheat_ton_infl",
    "Price_rice_ton_infl",
    "Price_corn_ton_infl"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing values
df[numeric_columns] = df[numeric_columns].fillna(
    df[numeric_columns].median()
)

print("Cleaning completed.")

Cleaning completed.


In [6]:
numeric_df = df.select_dtypes(include=np.number)

corr = numeric_df.corr()

corr

,Year,Price_wheat_ton,Price_rice_ton,Price_corn_ton,Inflation_rate,Price_wheat_ton_infl,Price_rice_ton_infl,Price_corn_ton_infl
Year,1.000000,0.610713,0.612335,0.605524,-0.982242,0.100766,0.127829,0.188468
Price_wheat_ton,0.610713,1.000000,0.746751,0.886950,-0.630338,0.838353,0.536634,0.762483
Price_rice_ton,0.612335,0.746751,1.000000,0.806890,-0.632364,0.527978,0.855870,0.668903
Price_corn_ton,0.605524,0.886950,0.806890,1.000000,-0.621045,0.692051,0.608042,0.884857
Inflation_rate,-0.982242,-0.630338,-0.632364,-0.621045,1.000000,-0.121663,-0.152221,-0.217739
Price_wheat_ton_infl,0.100766,0.838353,0.527978,0.692051,-0.121663,1.000000,0.600282,0.821343
Price_rice_ton_infl,0.127829,0.536634,0.855870,0.608042,-0.152221,0.600282,1.000000,0.707527
Price_corn_ton_infl,0.188468,0.762483,0.668903,0.884857,-0.217739,0.821343,0.707527,1.000000


In [7]:
numeric_df = df.select_dtypes(include=np.number)

corr = numeric_df.corr()

corr

,Year,Price_wheat_ton,Price_rice_ton,Price_corn_ton,Inflation_rate,Price_wheat_ton_infl,Price_rice_ton_infl,Price_corn_ton_infl
Year,1.000000,0.610713,0.612335,0.605524,-0.982242,0.100766,0.127829,0.188468
Price_wheat_ton,0.610713,1.000000,0.746751,0.886950,-0.630338,0.838353,0.536634,0.762483
Price_rice_ton,0.612335,0.746751,1.000000,0.806890,-0.632364,0.527978,0.855870,0.668903
Price_corn_ton,0.605524,0.886950,0.806890,1.000000,-0.621045,0.692051,0.608042,0.884857
Inflation_rate,-0.982242,-0.630338,-0.632364,-0.621045,1.000000,-0.121663,-0.152221,-0.217739
Price_wheat_ton_infl,0.100766,0.838353,0.527978,0.692051,-0.121663,1.000000,0.600282,0.821343
Price_rice_ton_infl,0.127829,0.536634,0.855870,0.608042,-0.152221,0.600282,1.000000,0.707527
Price_corn_ton_infl,0.188468,0.762483,0.668903,0.884857,-0.217739,0.821343,0.707527,1.000000


In [9]:
target="Price_wheat_ton"
X = df.drop(columns=[target])

y = df[target]

In [10]:
numeric_features = X.select_dtypes(include=np.number).columns

categorical_features = X.select_dtypes(
    include="object"
).columns

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

Train Model

In [12]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [13]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

In [14]:
model.fit(X_train, y_train)

print("Training completed.")

Training completed.


In [15]:
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)

rmse = np.sqrt(
    mean_squared_error(y_test, predictions)
)

r2 = r2_score(y_test, predictions)

print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)

MAE : 6.0940291666666795
RMSE: 8.7940969905402
R2  : 0.9797402131548127


In [16]:
joblib.dump(
    model,
    "../models/price_model.pkl"
)

print("Model saved successfully.")

Model saved successfully.
